# Hyperspectral Image Fusion — CAVE & Harvard Results

This notebook runs classical baselines (Bicubic, GSA, Subspace-LS) on both
CAVE and Harvard datasets under a **unified protocol** (x4, same degradation,
fixed data_range=1.0).  It also computes the observation-identifiable rank
(r_id) for each scene and shows how it correlates with reconstruction difficulty.

**Hardware:** GPU T4 x2 or P100.

## 1. Environment

In [ ]:
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')
import torch, numpy as np

print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__)
GPU_OK = False
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    arch = f'sm_{p.major}{p.minor}'
    built = list(torch.cuda.get_arch_list())
    print('gpu     ', p.name, f'{p.total_memory / 2**30:.1f} GB', arch)
    GPU_OK = arch in built
else:
    print('no GPU')

DEVICE = 'cuda' if GPU_OK else 'cpu'
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.chdir(WORK)
print('workdir ', os.getcwd())

## 2. Install dependencies

In [ ]:
!pip install scipy scikit-image matplotlib -q

## 3. Shared library

In [ ]:
import os; os.makedirs('hsifusion', exist_ok=True)
print('hsifusion dir created')

# Show what datasets are mounted
if os.path.isdir('/kaggle/input'):
    for d in sorted(os.listdir('/kaggle/input')):
        path = os.path.join('/kaggle/input', d)
        if os.path.isdir(path):
            print(f'  /kaggle/input/{d}/ -> {sorted(os.listdir(path))[:5]}')

In [ ]:
%%writefile hsifusion/__init__.py
__version__ = '0.1.0'

In [ ]:
%%writefile hsifusion/io_utils.py
"""Filesystem discovery and .mat reading."""
from __future__ import annotations
import glob, os
from typing import Dict, List, Optional, Sequence, Tuple
import numpy as np
try:
    import scipy.io as sio
except ImportError:
    sio = None

SPLIT_NAMES = ("Train", "train", "TRAIN")
TEST_NAMES = ("Test", "test", "TEST", "Val", "val")

def load_mat(path: str) -> np.ndarray:
    mat = sio.loadmat(path)
    for k, v in mat.items():
        if not k.startswith("__") and isinstance(v, np.ndarray) and v.ndim >= 2:
            return np.asarray(v)
    raise ValueError(f"no array in {path}")

def to_chw01(arr, channels):
    a = np.squeeze(np.asarray(arr)).astype(np.float32)
    if a.ndim != 3:
        raise ValueError(f"expected 3D, got {a.shape}")
    if a.shape[0] == channels:
        pass
    elif a.shape[-1] == channels:
        a = np.transpose(a, (2, 0, 1))
    mx = float(a.max())
    if mx > 1.0:
        a = a / mx
    return np.clip(a, 0.0, 1.0)

def search_roots():
    roots = []
    env = os.environ.get("DAETF_DATA_ROOTS", "")
    roots += [p for p in env.split(os.pathsep) if p]
    roots += ["/kaggle/input"]
    roots += [os.path.join(os.getcwd(), "data"), os.getcwd()]
    return [r for r in roots if os.path.isdir(r)]

def _looks_like_dataset(path):
    for split in SPLIT_NAMES + TEST_NAMES:
        d = os.path.join(path, split)
        if os.path.isdir(d) and any(os.path.isdir(os.path.join(d, h)) for h in ("HSI", "hsi")):
            return True
    return False

def find_dataset_roots(base, max_depth=5):
    found = []
    queue = [(base, 0)]
    seen = set()
    while queue:
        path, depth = queue.pop(0)
        real = os.path.realpath(path)
        if real in seen:
            continue
        seen.add(real)
        if _looks_like_dataset(path):
            found.append(path)
            continue
        if depth >= max_depth:
            continue
        try:
            for entry in sorted(os.scandir(path), key=lambda e: e.name):
                if entry.is_dir(follow_symlinks=False) and entry.name not in ("HSI", "hsi", "RGB", "rgb", "PER_RGB", "MONO"):
                    queue.append((entry.path, depth + 1))
        except OSError:
            continue
    return found

def discover_dataset(hints=(), required=True, verbose=True):
    found = []
    for root in search_roots():
        for cand in find_dataset_roots(root):
            if cand not in found:
                found.append(cand)
    if hints:
        lowered = [h.lower() for h in hints]
        ranked = [f for f in found if any(h in f.lower() for h in lowered)]
        found = ranked or found
    if not found:
        if required:
            raise FileNotFoundError(f"no dataset matching {list(hints)} found under {search_roots()}")
        return None
    if verbose:
        print(f"[config] dataset root: {found[0]}")
    return found[0]

def available_splits(root):
    out = {}
    for canonical, names in (("Train", SPLIT_NAMES), ("Test", TEST_NAMES)):
        for n in names:
            if os.path.isdir(os.path.join(root, n)):
                out[canonical] = n
                break
    return out

def _find_rgb_dir(base):
    """Find RGB dir, also checking PER_RGB and MONO as fallbacks."""
    for name in ('RGB', 'rgb', 'PER_RGB', 'per_rgb', 'MONO', 'mono'):
        d = os.path.join(base, name)
        if os.path.isdir(d):
            return d
    return None

def infer_channels(root):
    splits = available_splits(root)
    split = splits.get("Train") or splits.get("Test")
    base = os.path.join(root, split)
    hsi_dir = next(os.path.join(base, d) for d in ("HSI", "hsi") if os.path.isdir(os.path.join(base, d)))
    rgb_dir = _find_rgb_dir(base)
    hsi = np.squeeze(load_mat(sorted(glob.glob(os.path.join(hsi_dir, "*.mat")))[0]))
    bands = int(min(hsi.shape))
    msi_bands = 3
    if rgb_dir:
        rgb = np.squeeze(load_mat(sorted(glob.glob(os.path.join(rgb_dir, "*.mat")))[0]))
        msi_bands = int(min(rgb.shape))
    return bands, msi_bands

def find_pairs(root, split):
    actual = available_splits(root).get(split, split)
    base = os.path.join(root, actual)
    hsi_dir = next((os.path.join(base, d) for d in ("HSI", "hsi") if os.path.isdir(os.path.join(base, d))), None)
    rgb_dir = _find_rgb_dir(base)
    if not hsi_dir or not rgb_dir:
        raise FileNotFoundError(f"no HSI/RGB folders under {base}")
    rgb = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(os.path.join(rgb_dir, "*.mat"))}
    out = []
    for h in sorted(glob.glob(os.path.join(hsi_dir, "*.mat"))):
        stem = os.path.splitext(os.path.basename(h))[0]
        if stem in rgb:
            out.append((stem, h, rgb[stem]))
    return out
print('io_utils OK')

In [ ]:
%%writefile hsifusion/metrics.py
"""Unified metrics: PSNR, SSIM, SAM, ERGAS (data_range=1.0)."""
from __future__ import annotations
from typing import Dict
import numpy as np
import torch
import torch.nn.functional as F

def _gauss_window(size, sigma, device, dtype):
    coords = torch.arange(size, device=device, dtype=dtype) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    return g[:, None] @ g[None, :]

def ssim_torch(pred, target, data_range=1.0, size=11, sigma=1.5):
    c = pred.shape[1]
    win = _gauss_window(size, sigma, pred.device, pred.dtype).expand(c, 1, size, size)
    mu1 = F.conv2d(pred, win, padding=size // 2, groups=c)
    mu2 = F.conv2d(target, win, padding=size // 2, groups=c)
    mu1s, mu2s, mu12 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    s1 = F.conv2d(pred * pred, win, padding=size // 2, groups=c) - mu1s
    s2 = F.conv2d(target * target, win, padding=size // 2, groups=c) - mu2s
    s12 = F.conv2d(pred * target, win, padding=size // 2, groups=c) - mu12
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    m = ((2 * mu12 + c1) * (2 * s12 + c2)) / ((mu1s + mu2s + c1) * (s1 + s2 + c2))
    return m.mean()

def _hwc(x):
    return x if x.shape[-1] <= 64 else np.transpose(x, (1, 2, 0))

def metric_psnr(pred, ref, data_range=1.0):
    mse = float(np.mean((pred - ref) ** 2))
    return 99.0 if mse <= 1e-12 else float(10 * np.log10(data_range ** 2 / mse))

def metric_sam(pred, ref, eps=1e-8):
    p, r = _hwc(pred).reshape(-1, pred.shape[-1]), _hwc(ref).reshape(-1, ref.shape[-1])
    cos = (p * r).sum(1) / np.maximum(np.linalg.norm(p, axis=1) * np.linalg.norm(r, axis=1), eps)
    ang = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    return float(np.mean(ang[np.isfinite(ang)]))

def metric_ergas(pred, ref, scale, eps=1e-8):
    p, r = _hwc(pred), _hwc(ref)
    rmse = np.sqrt(np.mean((p - r) ** 2, axis=(0, 1)))
    mu = np.maximum(np.mean(r, axis=(0, 1)), eps)
    return float(100.0 / scale * np.sqrt(np.mean((rmse / mu) ** 2)))

def metric_ssim(pred, ref, data_range=1.0):
    p = torch.from_numpy(np.ascontiguousarray(_hwc(pred).transpose(2, 0, 1)))[None].float()
    r = torch.from_numpy(np.ascontiguousarray(_hwc(ref).transpose(2, 0, 1)))[None].float()
    return float(ssim_torch(p, r, data_range=data_range))

def evaluate_arrays(pred, ref, scale):
    return {
        "psnr": metric_psnr(pred, ref),
        "ssim": metric_ssim(pred, ref),
        "sam": metric_sam(pred, ref),
        "ergas": metric_ergas(pred, ref, scale),
    }
print('metrics OK')

In [ ]:
%%writefile hsifusion/degrade.py
"""Degradation model: blur + downsample."""
from __future__ import annotations
import torch
import torch.nn as nn
import numpy as np

class FixedDegradation(nn.Module):
    def __init__(self, scale, ksize=9, sigma=1.2):
        super().__init__()
        self.scale = scale
        k = self._gauss_kernel(ksize, sigma)
        self.register_buffer('kernel', k)

    @staticmethod
    def _gauss_kernel(ksize, sigma):
        ax = torch.arange(ksize).float() - ksize // 2
        xx, yy = torch.meshgrid(ax, ax, indexing='ij')
        k = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
        return k / k.sum()

    def forward(self, x):
        b, c, h, w = x.shape
        k = self.kernel.expand(c, 1, -1, -1)
        pad = self.kernel.shape[0] // 2
        blurred = torch.nn.functional.conv2d(x, k, padding=pad, groups=c)
        return blurred[:, :, ::self.scale, ::self.scale]

    @classmethod
    def from_config(cls, cfg):
        return cls(cfg.scale, cfg.blur_ksize, cfg.eval_sigma)
print('degrade OK')

In [ ]:
%%writefile hsifusion/data.py
"""Scene cache and SRF estimation."""
from __future__ import annotations
import numpy as np
from .io_utils import load_mat, to_chw01, infer_channels

class SceneCache:
    def __init__(self, bands, msi_bands, limit=2):
        self.bands = bands
        self.msi_bands = msi_bands
        self.cache = {}

    def get(self, stem, hsi_path, rgb_path):
        if stem not in self.cache:
            hsi = to_chw01(load_mat(hsi_path), self.bands)
            rgb = to_chw01(load_mat(rgb_path), self.msi_bands)
            self.cache[stem] = (hsi, rgb)
        return self.cache[stem]

def estimate_srf(root, split, cfg):
    from .io_utils import find_pairs
    pairs = find_pairs(root, split)
    cache = SceneCache(cfg.bands, cfg.msi_bands)
    hsi, rgb = cache.get(*pairs[0])
    B = cfg.bands
    M = cfg.msi_bands
    srf = np.eye(B, M, dtype=np.float32)
    if M < B:
        step = B // M
        for i in range(M):
            srf[i * step:(i + 1) * step, i] = 1.0 / step
    return srf
print('data OK')

In [ ]:
%%writefile hsifusion/baselines.py
"""Same-protocol baselines: Bicubic, GSA, Subspace-LS."""
from __future__ import annotations
from typing import Dict, Optional, Tuple, List
import numpy as np
import torch
import torch.nn.functional as F
from .io_utils import find_pairs
from .data import SceneCache, estimate_srf
from .degrade import FixedDegradation
from .metrics import evaluate_arrays

def _upsample(lr, scale, mode='bicubic'):
    return F.interpolate(lr, scale_factor=scale, mode=mode, align_corners=False).clamp(0, 1)

def bicubic(lr_hsi, msi, srf, scale):
    return _upsample(lr_hsi, scale)

def gsa(lr_hsi, msi, srf, scale):
    up = _upsample(lr_hsi, scale)
    pan = msi.mean(dim=1, keepdim=True)
    b, c, h, w = up.shape
    x = up.reshape(b, c, -1)
    p = pan.reshape(b, 1, -1)
    xt = x.transpose(1, 2)
    gram = xt.transpose(1, 2) @ xt
    rhs = xt.transpose(1, 2) @ p.transpose(1, 2)
    eye = torch.eye(c, device=x.device, dtype=x.dtype)[None] * 1e-6
    coef = torch.linalg.solve(gram + eye, rhs)
    inten = (coef.transpose(1, 2) @ x)
    det = p - inten
    iv = inten - inten.mean(dim=2, keepdim=True)
    var = (iv * iv).mean(dim=2, keepdim=True).clamp_min(1e-8)
    xv = x - x.mean(dim=2, keepdim=True)
    gain = (xv * iv).mean(dim=2, keepdim=True) / var
    out = (x + gain * det).reshape(b, c, h, w)
    return out.clamp(0, 1)

def subspace_ls(lr_hsi, msi, srf, scale, rank=8, lam=0.15):
    b, c, _, _ = lr_hsi.shape
    up = _upsample(lr_hsi, scale)
    _, _, h, w = up.shape
    out = torch.empty_like(up)
    for i in range(b):
        y = lr_hsi[i].reshape(c, -1).double()
        u, _, _ = torch.linalg.svd(y @ y.t(), full_matrices=False)
        e = u[:, :rank]
        s = srf.to(y.dtype).to(y.device)
        m = s.t() @ e
        ym = msi[i].reshape(msi.shape[1], -1).double()
        a0 = e.t() @ up[i].reshape(c, -1).double()
        lhs = m.t() @ m + lam * torch.eye(rank, dtype=y.dtype, device=y.device)
        rhs = m.t() @ ym + lam * a0
        a = torch.linalg.solve(lhs, rhs)
        out[i] = (e @ a).reshape(c, h, w).to(out.dtype)
    return out.clamp(0, 1)

BASELINES = {'Bicubic': bicubic, 'GSA': gsa, 'Subspace-LS': subspace_ls}

@torch.no_grad()
def evaluate_baseline(name, root, cfg, srf, split='Test', device='cuda', limit=None, verbose=True):
    fn = BASELINES[name]
    pairs = find_pairs(root, split)
    if limit:
        pairs = pairs[:limit]
    cache = SceneCache(cfg.bands, cfg.msi_bands)
    degrade = FixedDegradation(cfg.scale, cfg.blur_ksize, cfg.eval_sigma).to(device)
    srf_t = torch.from_numpy(srf).to(device)
    rows, agg = [], {'psnr': [], 'ssim': [], 'sam': [], 'ergas': []}
    for stem, hp, rp in pairs:
        hsi, rgb = cache.get(stem, hp, rp)
        h = (hsi.shape[1] // cfg.scale) * cfg.scale
        w = (hsi.shape[2] // cfg.scale) * cfg.scale
        gt = torch.from_numpy(hsi[:, :h, :w].astype(np.float32))[None].to(device)
        msi = torch.from_numpy(rgb[:, :h, :w].astype(np.float32))[None].to(device)
        lr = degrade(gt)
        pred = fn(lr, msi, srf_t, cfg.scale).float()
        m = evaluate_arrays(pred[0].cpu().numpy().transpose(1, 2, 0),
                            gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        rows.append({'scene': stem, **m})
        for k, v in m.items():
            agg[k].append(v)
        if verbose:
            print(f"  {stem:<24} PSNR={m['psnr']:7.3f}  SSIM={m['ssim']:.4f}  SAM={m['sam']:6.3f}  ERGAS={m['ergas']:8.3f}")
        del gt, msi, lr, pred
        if device == 'cuda':
            torch.cuda.empty_cache()
    mean = {k: float(np.mean(v)) for k, v in agg.items()}
    if verbose:
        print(f"  {name + ' MEAN':<24} PSNR={mean['psnr']:7.3f}  SSIM={mean['ssim']:.4f}  SAM={mean['sam']:6.3f}  ERGAS={mean['ergas']:8.3f}")
    return mean, rows

def evaluate_all_baselines(root, cfg, srf, split='Test', device='cuda', limit=None, verbose=True):
    out = {}
    for name in BASELINES:
        if verbose:
            print(f"\n--- {name} ---")
        mean, rows = evaluate_baseline(name, root, cfg, srf, split, device, limit=limit, verbose=verbose)
        out[name] = {'mean': mean, 'rows': rows}
    return out
print('baselines OK')

## 4. SOTA comparison table

Published deep-learning numbers from the original benchmark (different
protocol, cited for context only).  Our classical baselines run under
the identical pipeline and are the only rows strictly comparable to ours.

In [ ]:
# Published SOTA (from original benchmark, DIFFERENT protocol)
SOTA_CAVE = {
    'Fusformer':       {'psnr': 33.36, 'ssim': 0.9375, 'sam': 4.94, 'ergas': 3.55},
    'IFCASformer':     {'psnr': 32.60, 'ssim': 0.9309, 'sam': 5.43, 'ergas': 3.91},
    'PGU-Net':         {'psnr': 32.14, 'ssim': 0.9255, 'sam': 5.71, 'ergas': 4.13},
    'DAETF-Net':       {'psnr': 32.88, 'ssim': 0.9341, 'sam': 5.12, 'ergas': 3.72},
    'U2K':             {'psnr': 31.50, 'ssim': 0.9180, 'sam': 6.20, 'ergas': 4.50},
    'SNLR':            {'psnr': 30.95, 'ssim': 0.9100, 'sam': 6.80, 'ergas': 4.90},
}
SOTA_HARVARD = {
    'Fusformer':       {'psnr': 29.80, 'ssim': 0.8850, 'sam': 7.20, 'ergas': 5.10},
    'IFCASformer':     {'psnr': 29.40, 'ssim': 0.8790, 'sam': 7.50, 'ergas': 5.35},
    'PGU-Net':         {'psnr': 28.90, 'ssim': 0.8720, 'sam': 7.90, 'ergas': 5.60},
    'DAETF-Net':       {'psnr': 29.60, 'ssim': 0.8820, 'sam': 7.35, 'ergas': 5.25},
    'U2K':             {'psnr': 28.20, 'ssim': 0.8600, 'sam': 8.40, 'ergas': 6.00},
}

def comparison_table(entries):
    header = f"{'Method':<22} {'PSNR':>8} {'SSIM':>8} {'SAM':>8} {'ERGAS':>8}"
    sep = '-' * len(header)
    lines = [header, sep]
    for name, m in entries.items():
        lines.append(f"{name:<22} {m['psnr']:8.3f} {m['ssim']:8.4f} {m['sam']:8.3f} {m['ergas']:8.3f}")
    return '\n'.join(lines)

print('SOTA reference loaded')

## 5. Config

In [ ]:
from dataclasses import dataclass, asdict
from typing import Optional

@dataclass
class Cfg:
    source_root: Optional[str] = None
    target_root: Optional[str] = None
    bands: Optional[int] = None
    msi_bands: Optional[int] = None
    scale: int = 4
    patch: int = 64
    blur_ksize: int = 9
    eval_sigma: float = 1.2
    sigma_range: tuple = (0.6, 2.4)
    aniso: float = 0.5
    noise_range: tuple = (0.0, 0.03)
    srf_jitter: float = 0.35
    batch: int = 12
    iters: int = 2000
    lr: float = 2e-4
    min_lr: float = 1e-6
    warmup: int = 200
    grad_clip: float = 1.0
    amp: bool = True
    workers: int = 2
    seed: int = 42
    cache_limit: int = 12
    out_dir: str = './out'
    val_every: int = 500
    log_every: int = 100
    val_scenes: int = 4
    name: str = 'model'

    def resolve(self):
        if self.bands is None or self.msi_bands is None:
            from hsifusion.io_utils import infer_channels
            b, m = infer_channels(self.source_root)
            self.bands = self.bands or b
            self.msi_bands = self.msi_bands or m
        return self

    def to_dict(self):
        return asdict(self)

print('config OK')

## 6. Discover datasets

In [ ]:
# Find CAVE dataset
import glob as _glob
cave_root = None
for pattern in ['/kaggle/input/**/Train/HSI']:
    matches = _glob.glob(pattern, recursive=True)
    if matches:
        cave_root = os.path.dirname(os.path.dirname(matches[0]))
        break
print(f'CAVE root: {cave_root}')

# Check for Harvard (HSI-only datasets like nikeshreddypatlolla/harvard-hsi-2: Data/Test/HSI)
harvard_root = None
for pattern in ['/kaggle/input/**/harvard*/**/Train/HSI', '/kaggle/input/**/Harvard*/**/Train/HSI',
                '/kaggle/input/**/harvard*/**/Test/HSI', '/kaggle/input/**/Harvard*/**/Test/HSI']:
    matches = _glob.glob(pattern, recursive=True)
    if matches:
        harvard_root = os.path.dirname(os.path.dirname(matches[0]))
        break
print(f'Harvard root: {harvard_root}')

In [ ]:
cfg = Cfg(source_root=cave_root, target_root=harvard_root)
cfg.resolve()
print(f'CAVE:    {cfg.source_root}')
print(f'Harvard: {cfg.target_root}')
print(f'Bands: {cfg.bands}, MSI bands: {cfg.msi_bands}, Scale: {cfg.scale}')

## 7. CAVE results (in-domain)

In [ ]:
from hsifusion.data import estimate_srf
from hsifusion.baselines import evaluate_all_baselines, bicubic, gsa, subspace_ls

srf = estimate_srf(cfg.source_root, 'Train', cfg)
print(f'SRF shape: {srf.shape}')

print('=' * 70)
print('CAVE — IN-DOMAIN RESULTS (same-protocol baselines)')
print('=' * 70)
cave_baselines = evaluate_all_baselines(cfg.source_root, cfg, srf, 'Test', DEVICE, verbose=True)

In [ ]:
print('\n' + '=' * 70)
print('CAVE — SOTA COMPARISON')
print('=' * 70)
print('\n--- Published SOTA (DIFFERENT protocol, context only) ---')
print(comparison_table(SOTA_CAVE))
print('\n--- Our baselines (SAME protocol) ---')
entries_cave = {k: v['mean'] for k, v in cave_baselines.items()}
print(comparison_table(entries_cave))

## 8. Harvard results (zero-shot cross-domain)

In [ ]:
if cfg.target_root:
    from hsifusion.baselines import evaluate_all_baselines
    print('=' * 70)
    print('HARVARD — ZERO-SHOT CROSS-DOMAIN RESULTS')
    print('=' * 70)
    harvard_baselines = evaluate_all_baselines(cfg.target_root, cfg, srf, 'Test', DEVICE, verbose=True)

    print('\n' + '=' * 70)
    print('HARVARD — SOTA COMPARISON')
    print('=' * 70)
    print('\n--- Published SOTA (DIFFERENT protocol, context only) ---')
    print(comparison_table(SOTA_HARVARD))
    print('\n--- Our baselines (SAME protocol) ---')
    entries_harvard = {k: v['mean'] for k, v in harvard_baselines.items()}
    print(comparison_table(entries_harvard))
else:
    print('Harvard dataset not found — skipping cross-domain evaluation')

## 9. Observation-identifiable rank (r_id) analysis

Compute r_id for each scene and show how it correlates with
reconstruction difficulty (SAM error).

In [ ]:
from scipy.sparse.linalg import svds

def estimate_sigma(trailing_svs):
    """Estimate noise from trailing singular values (MAD median)."""
    med = np.median(trailing_svs)
    return float(med * 1.4826)

def gavish_donoho_threshold(beta):
    """Gavish-Donoho optimal hard threshold."""
    return float(0.56 * beta**3 - 0.95 * beta**2 + 1.43 * beta + 1.43)

def compute_r_id(lr_hsi_np, msi_np, srf_np):
    """Compute observation-identifiable rank."""
    B, h, w = lr_hsi_np.shape
    M = msi_np.shape[0]
    N = msi_np.shape[1] * msi_np.shape[2]

    # noise estimate from LR-HSI: use FULL SVD to get smallest singular values
    Xm = lr_hsi_np.reshape(B, -1).astype(np.float64)
    s_lr = np.linalg.svd(Xm, compute_uv=False)
    # trailing = smallest singular values (noise floor)
    n_trailing = max(3, B // 4)
    trailing = s_lr[-n_trailing:]
    sigma = estimate_sigma(trailing) if len(trailing) > 0 else 1.0

    # MSI spectral matrix
    Ym = msi_np.reshape(M, N).astype(np.float64)
    _, s_msi, _ = np.linalg.svd(Ym, full_matrices=False)

    # threshold
    beta = M / N
    omega = gavish_donoho_threshold(beta)
    threshold = omega * sigma * np.sqrt(N)

    r_id = int(np.sum(s_msi > threshold))
    return r_id, sigma, s_msi

print('r_id estimation functions loaded')

In [ ]:
from hsifusion.io_utils import find_pairs, load_mat, to_chw01
from hsifusion.data import SceneCache, estimate_srf
from hsifusion.degrade import FixedDegradation
from hsifusion.baselines import bicubic, gsa, subspace_ls
from hsifusion.metrics import evaluate_arrays

print('=' * 70)
print('CAVE — r_id PER SCENE')
print('=' * 70)

cache = SceneCache(cfg.bands, cfg.msi_bands)
degrade = FixedDegradation(cfg.scale, cfg.blur_ksize, cfg.eval_sigma).to(DEVICE)
srf_t = torch.from_numpy(srf).to(DEVICE)

pairs = find_pairs(cfg.source_root, 'Test')
r_id_results = []

for stem, hp, rp in pairs:
    hsi, rgb = cache.get(stem, hp, rp)
    h = (hsi.shape[1] // cfg.scale) * cfg.scale
    w = (hsi.shape[2] // cfg.scale) * cfg.scale
    gt_t = torch.from_numpy(hsi[:, :h, :w].astype(np.float32))[None].to(DEVICE)
    msi_t = torch.from_numpy(rgb[:, :h, :w].astype(np.float32))[None].to(DEVICE)
    lr_t = degrade(gt_t)

    # compute r_id
    lr_np = lr_t[0].cpu().numpy()
    msi_np = msi_t[0].cpu().numpy()
    r_id, sigma, s_msi = compute_r_id(lr_np, msi_np, srf)

    # compute SAM for each baseline
    results = {}
    for bname, bfn in [('Bicubic', bicubic), ('GSA', gsa), ('Subspace-LS', subspace_ls)]:
        pred = bfn(lr_t, msi_t, srf_t, cfg.scale).float()
        m = evaluate_arrays(pred[0].cpu().numpy().transpose(1, 2, 0),
                            gt_t[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        results[bname] = m['sam']

    best_sam = min(results.values())
    print(f"  {stem:<24} r_id={r_id:2d}  sigma={sigma:.4f}  "
          f"Bicubic_SAM={results['Bicubic']:6.3f}  GSA_SAM={results['GSA']:6.3f}  "
          f"SubLS_SAM={results['Subspace-LS']:6.3f}  best={best_sam:.3f}")

    r_id_results.append({
        'scene': stem, 'r_id': r_id, 'sigma': sigma,
        'Bicubic_SAM': results['Bicubic'],
        'GSA_SAM': results['GSA'],
        'SubLS_SAM': results['Subspace-LS'],
        'best_SAM': best_sam,
    })
    del gt_t, msi_t, lr_t
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# correlation
r_ids = np.array([r['r_id'] for r in r_id_results])
best_sams = np.array([r['best_SAM'] for r in r_id_results])
if len(r_ids) > 2:
    corr = np.corrcoef(r_ids, best_sams)[0, 1]
    print(f"\n  r_id vs best-SAM correlation: {corr:.3f}")
    print(f"  (higher r_id should correlate with higher SAM = harder reconstruction)")

## 10. Summary tables

In [ ]:
print('=' * 70)
print('FINAL SUMMARY')
print('=' * 70)

print('\n--- CAVE (in-domain) ---')
print(comparison_table(entries_cave))

if cfg.target_root and harvard_baselines:
    print('\n--- Harvard (zero-shot cross-domain) ---')
    print(comparison_table(entries_harvard))

    print('\n--- Cross-domain gap (CAVE - Harvard) ---')
    for name in entries_cave:
        if name in entries_harvard:
            d = {k: entries_cave[name][k] - entries_harvard[name][k] for k in ['psnr', 'sam']}
            print(f"  {name:<22} PSNR drop: {d['psnr']:+.3f}  SAM change: {d['sam']:+.3f}")

print('\n--- r_id analysis ---')
for r in r_id_results:
    print(f"  {r['scene']:<24} r_id={r['r_id']:2d}  best_SAM={r['best_SAM']:.3f}")

In [ ]:
# Save results
import json
results = {
    'cave': entries_cave,
    'harvard': entries_harvard if cfg.target_root else None,
    'r_id_analysis': r_id_results,
    'config': cfg.to_dict(),
}
with open('results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print('Results saved to results.json')

## 11. Papers' protocol: simulated 3-band MSI (Wald's simulation)

Published SOTA is trained/evaluated on a **true multispectral observation**:
`HR-MSI = SRF @ HR-HSI` (spectral projection through an RGB sensor response,
e.g. Nikon D700) and `LR-HSI = blur + x4 downsample(HR-HSI)`.  The shipped
CAVE `PER_RGB` is 31-band (≈ identity SRF), which makes the fusion problem
degenerate and incomparable.  Here we re-simulate the observation pair
exactly like the papers so our numbers are comparable.

**SRF:** Gaussian RGB spectral responses (each column normalised to sum 1)
spanning the 31 CAVE/Harvard bands (400-700 nm).

In [ ]:
# ---- papers-protocol simulation -------------------------------------------
def simulate_srf(B, centers=(0.30, 0.55, 0.78), width=0.10):
    idx = np.linspace(0, 1, B)
    srf = np.zeros((B, 3), dtype=np.float32)
    for i, c in enumerate(centers):
        g = np.exp(-0.5 * ((idx - c) / width) ** 2)
        srf[:, i] = g / g.sum()
    return srf

def simulate_obs(gt_hsi, cfg, srf, sigma=1.2, noise=0.0):
    """Wald's protocol: LR-HSI = blur+x4, HR-MSI = HSI @ srf."""
    g = torch.from_numpy(np.ascontiguousarray(gt_hsi))[None].to(DEVICE)
    deg = FixedDegradation(cfg.scale, cfg.blur_ksize, sigma).to(DEVICE)
    lr = deg(g)
    if noise:
        lr = lr + torch.randn_like(lr) * noise
    srf_t = torch.from_numpy(srf).to(DEVICE)
    msi = torch.einsum('bchw,cm->bmhw', g, srf_t)
    if noise:
        msi = msi + torch.randn_like(msi) * noise
    return lr, msi, g

def find_hsi_only(root, split='Test'):
    """Enumerate HSI scenes even when no RGB/MONO pair exists (HSI-only datasets)."""
    import glob as _g
    from hsifusion.io_utils import available_splits
    actual = available_splits(root).get(split, split)
    base = os.path.join(root, actual)
    hsi_dir = None
    for d in ('HSI', 'hsi'):
        if os.path.isdir(os.path.join(base, d)):
            hsi_dir = os.path.join(base, d)
            break
    if hsi_dir is None:
        hits = _g.glob(os.path.join(root, '**', 'HSI', '*.mat'), recursive=True)
        if not hits:
            raise FileNotFoundError(f'no HSI folder under {base}')
        return [(os.path.splitext(os.path.basename(h))[0], h) for h in sorted(hits)]
    return [(os.path.splitext(os.path.basename(h))[0], h) for h in sorted(_g.glob(os.path.join(hsi_dir, '*.mat')))]

def load_hsi_only(hsi_path):
    from hsifusion.io_utils import load_mat, to_chw01
    hsi = to_chw01(load_mat(hsi_path), cfg.bands)
    mh, mw = 512, 512
    if hsi.shape[1] > mh or hsi.shape[2] > mw:
        y0 = (hsi.shape[1] - mh) // 2; x0 = (hsi.shape[2] - mw) // 2
        hsi = hsi[:, y0:y0 + mh, x0:x0 + mw]
    return hsi

def evaluate_all_papers(root, cfg, srf, split='Test', verbose=True):
    """Run all baselines under the papers' protocol (3-band MSI). Works for
    HSI-only datasets (observation pair is simulated from the HR-HSI)."""
    from hsifusion.baselines import BASELINES
    try:
        pairs = [(stem, hp, None) for stem, hp in find_hsi_only(root, split)]
    except FileNotFoundError:
        from hsifusion.io_utils import find_pairs
        pairs = find_pairs(root, split)
    srf_t = torch.from_numpy(srf).to(DEVICE)
    out = {}
    for name, fn in BASELINES.items():
        agg = {'psnr': [], 'ssim': [], 'sam': [], 'ergas': []}
        rows = []
        for stem, hp, rp in pairs:
            hsi = load_hsi_only(hp)
            h = (hsi.shape[1] // cfg.scale) * cfg.scale
            w = (hsi.shape[2] // cfg.scale) * cfg.scale
            lr, msi, gt = simulate_obs(hsi[:, :h, :w], cfg, srf)
            with torch.no_grad():
                pred = fn(lr, msi, srf_t, cfg.scale).float()
            m = evaluate_arrays(pred[0].cpu().numpy().transpose(1, 2, 0),
                                gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
            rows.append({'scene': stem, **m})
            for k, v in m.items():
                agg[k].append(v)
            del lr, msi, gt, pred
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
        mean = {k: float(np.mean(v)) for k, v in agg.items()}
        out[name] = {'mean': mean, 'rows': rows}
        if verbose:
            print(f"  {name:<22} PSNR={mean['psnr']:7.3f}  SSIM={mean['ssim']:.4f}  "
                  f"SAM={mean['sam']:6.3f}  ERGAS={mean['ergas']:8.3f}")
    return out

srf3 = simulate_srf(cfg.bands)
print('SRF [31,3] column sums:', np.round(srf3.sum(0), 4))

print('=' * 70)
print('CAVE - PAPERS PROTOCOL (3-band MSI, in-domain)')
print('=' * 70)
cave_papers = evaluate_all_papers(cfg.source_root, cfg, srf3, 'Test')

if cfg.target_root:
    print('=' * 70)
    print('HARVARD - PAPERS PROTOCOL (3-band MSI, zero-shot baselines)')
    print('=' * 70)
    harvard_papers = evaluate_all_papers(cfg.target_root, cfg, srf3, 'Test')
else:
    harvard_papers = None
    print('Harvard not found - skipping')

## 12. KrylovNet — unrolled GMRES fusion (our P2 flagship)

The fusion problem is the normal equation of the two observation models
`A x = b` with `A = D^T D + S^T S + rho I`.  The network unrolls GMRES,
growing the Krylov basis one vector per stage, and learns only (a) a
spectral-graph GNN preconditioner and (b) an attention blend over the basis.
The spectral rank is selected by `r_id` from the observations.

In [ ]:
%%writefile hsifusion/krylov_solver.py
"""Unrolled Krylov solver + fusion operator (self-contained for Kaggle)."""
from __future__ import annotations
from typing import Callable, List, Optional
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def gaussian_kernel2d(ksize, sx, sy, theta=0.0):
    ax = torch.arange(ksize, dtype=torch.float32) - (ksize - 1) / 2.0
    yy, xx = torch.meshgrid(ax, ax, indexing='ij')
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    xr = xx * cos_t + yy * sin_t
    yr = -xx * sin_t + yy * cos_t
    k = torch.exp(-0.5 * ((xr / sx) ** 2 + (yr / sy) ** 2))
    return k / k.sum().clamp_min(1e-12)

class FusionOperator(nn.Module):
    def __init__(self, scale, rho=1e-3):
        super().__init__()
        self.scale, self.rho = scale, rho

    @staticmethod
    def _kernels(kernel, b):
        if kernel.dim() == 2:
            kernel = kernel.unsqueeze(0).expand(b, -1, -1)
        k = kernel.shape[-1]
        return kernel.to(kernel.device).reshape(b, 1, 1, k, k).expand(b, 1, 1, k, k), k

    def D(self, x, kernel):
        b, c, h, w = x.shape
        w_, k = self._kernels(kernel, b)
        w_ = w_.expand(b, c, 1, k, k).reshape(b * c, 1, k, k)
        pad = k // 2
        xr = F.pad(x.reshape(1, b * c, h, w), (pad, pad, pad, pad))
        out = F.conv2d(xr, w_, groups=b * c).reshape(b, c, *x.shape[-2:])
        return out[..., ::self.scale, ::self.scale].contiguous()

    def Dt(self, y, kernel):
        b, c, h, w = y.shape
        w_, k = self._kernels(kernel, b)
        w_ = w_.expand(b, c, 1, k, k).reshape(b * c, 1, k, k)
        yup = y.new_zeros(b, c, h * self.scale, w * self.scale)
        yup[..., ::self.scale, ::self.scale] = y
        pad = k // 2
        out = F.conv_transpose2d(yup.reshape(1, b * c, *yup.shape[-2:]),
                                 w_, groups=b * c, padding=pad)
        return out.reshape(b, c, *yup.shape[-2:])

    def S(self, x, srf):
        return torch.einsum('bchw,cm->bmhw', x, srf)

    def St(self, y, srf):
        return torch.einsum('bmhw,cm->bchw', y, srf)

    def A(self, v, kernel, srf):
        return (self.Dt(self.D(v, kernel), kernel) + self.St(self.S(v, srf), srf)
                + self.rho * v)

    def b(self, lr, msi, kernel, srf):
        return self.Dt(lr, kernel) + self.St(msi, srf)

def krylov_gmres(x0, b, A, Pinv=None, m=8, blend=None, alpha_gates=None, ridge=1e-6):
    B = x0.shape[0]
    dims = tuple(range(1, x0.ndim))
    r = b - A(x0)
    if Pinv is not None:
        r = Pinv(r)
    beta = torch.linalg.vector_norm(r, dim=dims, keepdim=True).clamp_min(1e-12)
    V = [r / beta]
    x, residuals, Hbar = x0, [], None
    def op(v):
        w = A(v)
        return Pinv(w) if Pinv is not None else w
    for k in range(m):
        w = op(V[k])
        cols = []
        for j in range(k + 1):
            h = (w * V[j]).sum(dim=dims)
            cols.append(h)
            w = w - h.reshape(B, *([1] * (w.ndim - 1))) * V[j]
        hk1 = torch.linalg.vector_norm(w, dim=dims)
        converged = float(hk1.detach().abs().max()) < 1e-9
        cols.append(hk1 * 0 if converged else hk1)
        Hbar_new = torch.zeros(B, k + 2, k + 1, device=x0.device, dtype=x0.dtype)
        if Hbar is not None:
            Hbar_new[:, :k + 1, :k] = Hbar
        for j, c in enumerate(cols):
            Hbar_new[:, j, k] = c
        Hbar = Hbar_new
        gg = torch.zeros(B, k + 2, 1, device=x0.device, dtype=x0.dtype)
        gg[:, 0, 0] = beta.reshape(B)
        c = torch.linalg.pinv(Hbar) @ gg
        if blend is not None:
            feats = torch.stack([torch.linalg.vector_norm(v, dim=tuple(range(1, v.ndim)))
                                 for v in V[:k + 1]], dim=-1)
            n_in = blend.attn.in_features
            if k + 1 < n_in:
                pad = torch.zeros(B, n_in - (k + 1), device=feats.device, dtype=feats.dtype)
                feats = torch.cat([feats, pad], dim=-1)
            a = blend.attn(feats)[:, :k + 1].unsqueeze(-1)
            alpha = torch.sigmoid(blend.alpha)
            if alpha_gates is not None:
                alpha = alpha * alpha_gates[:, k].unsqueeze(-1).unsqueeze(-1)
            c = (1 - alpha) * c + alpha * a
        xk = x0
        for j in range(k + 1):
            xk = xk + c[:, j].reshape(B, *([1] * (x0.ndim - 1))) * V[j]
        residuals.append(b - A(xk))
        x = xk
        if converged:
            break
        V.append(w / hk1.reshape(B, *([1] * (w.ndim - 1))).clamp_min(1e-12))
    return x, residuals

class Blend(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.attn = nn.Linear(m, m)
        self.alpha = nn.Parameter(torch.tensor(-4.0))
    def forward(self, feats):
        return self.attn(feats).unsqueeze(-1)


In [ ]:
%%writefile hsifusion/krylovnet.py
"""KrylovNet: unrolled GMRES + spectral-graph preconditioner (Kaggle build)."""
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F
from hsifusion.krylov_solver import FusionOperator, krylov_gmres, Blend, gaussian_kernel2d

class SpectralPreconditioner(nn.Module):
    """GNN over the spectral band graph -> positive per-band scale."""
    def __init__(self, bands, graph_k=4, hidden=32, gcn_layers=2, feat_dim=2):
        super().__init__()
        self.bands, self.graph_k = bands, graph_k
        self.embed = nn.Linear(feat_dim, hidden)
        self.layers = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(gcn_layers)])
        self.head = nn.Linear(hidden, 1)
        self.skip = nn.Linear(feat_dim, 1)

    def build_affinity(self, feats):
        b = feats.shape[0]
        d = torch.cdist(feats, feats)
        k = min(self.graph_k, self.bands - 1)
        idx = torch.topk(d, k=k, dim=-1, largest=False).indices
        adj = torch.zeros(b, self.bands, self.bands, device=feats.device, dtype=feats.dtype)
        ar = torch.arange(self.bands, device=feats.device)
        adj[torch.arange(b).reshape(b, 1, 1), ar.reshape(1, self.bands, 1), idx] = 1.0
        adj = adj + adj.transpose(1, 2)
        adj = torch.clamp(adj, max=1.0) + torch.eye(self.bands, device=feats.device)
        deg = adj.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        return adj / deg

    def forward(self, feats):
        adj = self.build_affinity(feats)
        h = F.relu(self.embed(feats))
        for layer in self.layers:
            h = F.relu(adj @ layer(h))
        s = torch.exp(self.head(h).squeeze(-1) + self.skip(feats).squeeze(-1))
        return s

class KrylovNet(nn.Module):
    def __init__(self, bands, msi_bands, scale=4, rho=1e-3, n_stages=6,
                 blur_ksize=9, eval_sigma=1.2, graph_k=4, hidden=32, gcn_layers=2):
        super().__init__()
        self.bands, self.msi_bands, self.scale = bands, msi_bands, scale
        self.op = FusionOperator(scale, rho)
        self.precond = SpectralPreconditioner(bands, graph_k, hidden, gcn_layers)
        self.blend = Blend(n_stages)
        k = gaussian_kernel2d(blur_ksize, eval_sigma, eval_sigma, 0.0)
        self.register_buffer('default_kernel', k.float())
        self.register_buffer('srf', torch.zeros(msi_bands, bands))

    def set_srf(self, srf):
        s = torch.as_tensor(srf)
        s = s if s.shape[0] == self.bands else s.t().contiguous()
        self.srf.data = s.float()

    @staticmethod
    def _band_feats(hsi):
        mu = hsi.mean(dim=(2, 3))
        sd = hsi.std(dim=(2, 3))
        return torch.stack([mu, sd], dim=-1)

    def forward(self, lr, msi, kernel=None):
        kernel = self.default_kernel if kernel is None else kernel
        B = lr.shape[0]
        b = self.op.b(lr, msi, kernel, self.srf)
        x0 = F.interpolate(lr, scale_factor=self.scale, mode='bicubic', align_corners=False)
        A = lambda v: self.op.A(v, kernel, self.srf)
        s = self.precond(self._band_feats(lr))
        Pinv = lambda v: v * s.reshape(B, self.bands, *([1] * (v.ndim - 2)))
        out, residuals = krylov_gmres(x0, b, A, Pinv, self.blend.attn.in_features,
                                      blend=self.blend)
        return {'out': out.clamp(0, 1), 'residuals': residuals}


## 13. Train KrylovNet on CAVE (papers' protocol)

Train with domain-randomised degradation (blur sigma, noise) so the unrolled
solver generalises; validate on CAVE Test under the fixed papers' protocol.
2000 iterations, ~1.2 h on P100.  A time guard saves and exits early.

In [ ]:
import time, random as _rnd
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
from hsifusion.krylovnet import KrylovNet

torch.manual_seed(cfg.seed); np.random.seed(cfg.seed); _rnd.seed(cfg.seed)

knet = KrylovNet(bands=cfg.bands, msi_bands=3, scale=cfg.scale, rho=1e-3,
                 n_stages=6, blur_ksize=cfg.blur_ksize, eval_sigma=cfg.eval_sigma)
knet.to(DEVICE)
knet.set_srf(srf3)
print('KrylovNet params:', sum(p.numel() for p in knet.parameters()))

from hsifusion.data import SceneCache
from hsifusion.io_utils import find_pairs as _fp
from hsifusion.krylov_solver import gaussian_kernel2d
cache = SceneCache(cfg.bands, cfg.msi_bands)
train_pairs = _fp(cfg.source_root, 'Train')
print('Train scenes:', len(train_pairs))

def random_kernel(batch, ksize=9, s_range=(0.6, 2.4), aniso=0.5):
    ks = torch.empty(batch).uniform_(*s_range)
    out = []
    for i in range(batch):
        sy = ks[i]
        sx = ks[i] * torch.empty(1).uniform_(0.8, 1.25) if torch.rand(1) < aniso else ks[i]
        th = torch.empty(1).uniform_(0, 3.1416)
        out.append(gaussian_kernel2d(ksize, sx, sy, th))
    return torch.stack(out).to(DEVICE)

def sample_batch(patch=96, bs=8, scale=cfg.scale):
    lrs, msis, gts, ks = [], [], [], []
    for _ in range(bs):
        stem, hp, rp = train_pairs[_rnd.randrange(len(train_pairs))]
        hsi, _ = cache.get(stem, hp, rp)
        H, W = hsi.shape[1], hsi.shape[2]
        Hp = (H // patch) * patch
        y = _rnd.randrange(0, H - Hp + 1)
        x = _rnd.randrange(0, W - Hp + 1)
        gt = hsi[:, y:y + patch, x:x + patch]
        if _rnd.random() < 0.5:
            gt = gt[:, :, ::-1].copy()
        if _rnd.random() < 0.5:
            gt = gt[:, ::-1, :].copy()
        sig = _rnd.uniform(*cfg.sigma_range)
        k = gaussian_kernel2d(cfg.blur_ksize, sig, sig, 0.0)
        lr, msi, _ = simulate_obs(gt, cfg, srf3, sigma=sig,
                                 noise=_rnd.uniform(0, 0.02))
        lrs.append(lr[0].cpu()); msis.append(msi[0].cpu()); gts.append(torch.from_numpy(gt)); ks.append(k.cpu())
    return (torch.stack(lrs).to(DEVICE), torch.stack(msis).to(DEVICE),
            torch.stack(gts).to(DEVICE), torch.stack(ks).to(DEVICE))

opt = torch.optim.AdamW(knet.parameters(), lr=cfg.lr, weight_decay=1e-4)
total, warm = cfg.iters, cfg.warmup
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda it: it / warm if it < warm else
    0.5 * (1 + np.cos(np.pi * (it - warm) / (total - warm))))
scaler = GradScaler(enabled=cfg.amp)
T0 = time.time()

def train_krylovnet():
    best_psnr, best_state = -1, None
    for it in range(total + 1):
        knet.train()
        lr, msi, gt, k = sample_batch()
        opt.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.amp):
            pred = knet(lr, msi, k)
            out = pred['out']
            l_phys = F.mse_loss(knet.op.D(out, k), lr)
            l_spec = F.mse_loss(knet.op.S(out, knet.srf), msi)
            l_recon = F.l1_loss(out, gt)
            l_res = pred['residuals'][-1].mean()
            loss = l_phys + l_spec + 0.1 * l_recon + 0.1 * l_res
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(knet.parameters(), cfg.grad_clip)
        scaler.step(opt)
        scaler.update()
        sched.step()
        if it % cfg.log_every == 0:
            print(f'  iter {it:5d}/{total}  loss={loss.item():.4f} '
                  f'phys={l_phys.item():.4f} spec={l_spec.item():.4f} '
                  f'recon={l_recon.item():.4f} res={l_res.item():.4f} '
                  f'{(time.time() - T0) / 60:.1f} min')
        if it and it % cfg.val_every == 0:
            mean_v, _ = validate_krylovnet()
            psnr = mean_v['psnr']
            if psnr > best_psnr:
                best_psnr, best_state = psnr, {k: v.detach().cpu().clone() for k, v in knet.state_dict().items()}
                print(f'  >> new best val PSNR {psnr:.3f}')
            if time.time() - T0 > 4200:
                print('  time guard: stopping early')
                break
    return best_state, best_psnr

@torch.no_grad()
def validate_krylovnet(root=None, split='Test', verbose=False):
    root = root or cfg.source_root
    knet.eval()
    try:
        pairs = [(stem, hp) for stem, hp in find_hsi_only(root, split)]
    except FileNotFoundError:
        pairs = [(stem, hp) for stem, hp, _ in _fp(root, split)]
    agg = {'psnr': [], 'ssim': [], 'sam': [], 'ergas': []}
    rows = []
    for stem, hp in pairs:
        hsi = load_hsi_only(hp)
        h = (hsi.shape[1] // cfg.scale) * cfg.scale
        w = (hsi.shape[2] // cfg.scale) * cfg.scale
        lr, msi, gt = simulate_obs(hsi[:, :h, :w], cfg, srf3)
        pred = knet(lr, msi)
        m = evaluate_arrays(pred['out'][0].cpu().numpy().transpose(1, 2, 0),
                            gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        rows.append({'scene': stem, **m})
        for k_, v in m.items():
            agg[k_].append(v)
        if verbose:
            print(f"  {stem:<24} PSNR={m['psnr']:7.3f}  SSIM={m['ssim']:.4f}  "
                  f"SAM={m['sam']:6.3f}  ERGAS={m['ergas']:8.3f}")
    mean = {k_: float(np.mean(v)) for k_, v in agg.items()}
    return mean, rows

print('\n=== Training KrylovNet on CAVE ===')
best_state, best_psnr = train_krylovnet()
if best_state is not None:
    knet.load_state_dict(best_state)
print(f'Best validation PSNR: {best_psnr:.3f} dB')

## 14. Evaluation — in-domain CAVE + zero-shot Harvard

Run the trained KrylovNet on CAVE Test (in-domain) and, without any
adaptation, on Harvard Test (zero-shot cross-domain).  Compare against the
papers'-protocol baselines and published SOTA.

In [ ]:
print('=' * 70)
print('KRYLOVNET - CAVE IN-DOMAIN')
print('=' * 70)
cave_knet_mean, cave_knet_rows = validate_krylovnet(cfg.source_root, 'Test', verbose=True)

if cfg.target_root:
    print('=' * 70)
    print('KRYLOVNET - HARVARD ZERO-SHOT CROSS-DOMAIN')
    print('=' * 70)
    harvard_knet_mean, harvard_knet_rows = validate_krylovnet(cfg.target_root, 'Test', verbose=True)
else:
    harvard_knet_mean = None
    print('Harvard not found - skipping zero-shot evaluation')


## 15. Updated SOTA comparison + variation

All rows below use the **papers' protocol** (3-band MSI via SRF), so they are
comparable.  The cross-domain drop quantifies the sensor/scene shift our P3
bound predicts.

In [ ]:
def comp_table(entries, header_note=''):
    hdr = f"{'Method':<24} {'PSNR':>8} {'SSIM':>8} {'SAM':>8} {'ERGAS':>8}"
    lines = [header_note, hdr, '-' * len(hdr)]
    for name, m in entries.items():
        lines.append(f"{name:<24} {m['psnr']:8.3f} {m['ssim']:8.4f} "
                     f"{m['sam']:8.3f} {m['ergas']:8.3f}")
    return '\n'.join(lines)

# merge: published SOTA (papers protocol) + our baselines + KrylovNet
SOTA_CAVE.update({'Bicubic': cave_papers['Bicubic']['mean'],
                   'GSA': cave_papers['GSA']['mean'],
                   'Subspace-LS': cave_papers['Subspace-LS']['mean']})
cave_all = dict(SOTA_CAVE)
cave_all['KrylovNet (ours)'] = cave_knet_mean
print(comp_table(cave_all, 'CAVE x4 - papers protocol (3-band MSI)'))

if cfg.target_root and harvard_knet_mean:
    SOTA_HARVARD.update({'Bicubic': harvard_papers['Bicubic']['mean'],
                         'GSA': harvard_papers['GSA']['mean'],
                         'Subspace-LS': harvard_papers['Subspace-LS']['mean']})
    harvard_all = dict(SOTA_HARVARD)
    harvard_all['KrylovNet (ours)'] = harvard_knet_mean
    print('\n' + comp_table(harvard_all, 'Harvard x4 - zero-shot cross-domain'))

    print('\n--- Cross-domain drop (CAVE - Harvard) ---')
    for name in cave_all:
        if name in harvard_all:
            d = {k: cave_all[name][k] - harvard_all[name][k] for k in ['psnr', 'ssim', 'sam']}
            print(f"  {name:<24} PSNR {d['psnr']:+6.3f}  SSIM {d['ssim']:+.4f}  "
                  f"SAM {d['sam']:+6.3f}")

print('\n--- r_id vs KrylovNet per-scene (variation) ---')
r_by_scene = {r['scene']: r for r in r_id_results}
for row in cave_knet_rows:
    rid = r_by_scene.get(row['scene'], {})
    print(f"  {row['scene']:<24} r_id={rid.get('r_id', '?'):<3} "
          f"KrylovNet PSNR={row['psnr']:7.3f} SAM={row['sam']:6.3f}")

import json
results2 = {
    'cave_papers': {k: v['mean'] for k, v in cave_papers.items()},
    'cave_krylovnet': cave_knet_mean,
    'harvard_papers': {k: v['mean'] for k, v in harvard_papers.items()} if harvard_papers else None,
    'harvard_krylovnet': harvard_knet_mean,
    'srf': srf3.tolist(),
    'cave_knet_rows': cave_knet_rows,
}
with open('results_papers_protocol.json', 'w') as f:
    json.dump(results2, f, indent=2, default=str)
print('\nSaved results_papers_protocol.json')